# 🌐 Multivariate Calculus & Gradients

Welcome to the world of multiple dimensions! This is where machine learning really happens.

## Why Multivariate Calculus?

- Neural networks have **millions of parameters**
- We need to optimize **all of them simultaneously**
- **Gradients** tell us how to update each parameter
- **Backpropagation** = Chain rule in multiple dimensions

## What You'll Learn
1. Gradients and their geometric meaning
2. Jacobian matrices
3. Multivariate chain rule
4. Hessian matrices and second derivatives
5. Backpropagation visualization

In [ ]:
# Import our tools
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. The Gradient Vector

For a function $f(x, y)$, the gradient is:

$$\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x} \\ \frac{\partial f}{\partial y} \end{bmatrix}$$

**Key insight:** The gradient points in the direction of **steepest ascent**!

In [ ]:
# Define a simple 2D function
def f(x, y):
    return x**2 + 2*y**2

# Gradient components
def grad_f(x, y):
    df_dx = 2*x
    df_dy = 4*y
    return np.array([df_dx, df_dy])

# Create meshgrid
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

# Create figure with 3D and 2D views
fig = plt.figure(figsize=(18, 7))

# 3D surface plot
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.7, edgecolor='none')

# Add some gradient vectors in 3D
points = [(-2, -1), (-1, 1), (1, -1), (2, 1)]
for px, py in points:
    grad = grad_f(px, py)
    ax1.quiver(px, py, f(px, py), grad[0], grad[1], 0, 
               color='red', arrow_length_ratio=0.3, linewidth=2.5)

ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_zlabel('f(x,y)', fontsize=12)
ax1.set_title('Surface with Gradient Vectors', fontsize=13, fontweight='bold')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# 2D contour plot with gradient field
ax2 = fig.add_subplot(122)
contour = ax2.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
contour_lines = ax2.contour(X, Y, Z, levels=20, colors='black', alpha=0.3, linewidths=0.5)
ax2.clabel(contour_lines, inline=True, fontsize=8)

# Plot gradient field
step = 8
for i in range(0, len(x), step):
    for j in range(0, len(y), step):
        grad = grad_f(X[j, i], Y[j, i])
        grad_norm = np.linalg.norm(grad)
        if grad_norm > 0:
            grad_normalized = grad / grad_norm * 0.3
            ax2.arrow(X[j, i], Y[j, i], grad_normalized[0], grad_normalized[1],
                     head_width=0.15, head_length=0.1, fc='red', ec='red', 
                     alpha=0.7, linewidth=1.5)

ax2.plot(0, 0, 'r*', markersize=20, label='Minimum (∇f = 0)')
ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('y', fontsize=12)
ax2.set_title('Gradient Field (red arrows)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.set_aspect('equal')
fig.colorbar(contour, ax=ax2)

plt.suptitle('🎯 Gradient: Direction of Steepest Ascent', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("Red arrows point uphill (gradient direction)")
print("To minimize, go opposite direction: -∇f")
print(f"\nAt point (2, 1): ∇f = {grad_f(2, 1)}")

## 2. Directional Derivatives

The **directional derivative** tells us how fast $f$ changes in a specific direction.

$$D_{\mathbf{v}}f = \nabla f \cdot \mathbf{v}$$

Where $\mathbf{v}$ is a unit vector indicating direction.

In [ ]:
# Point of interest
point = np.array([1.5, 1.0])
gradient = grad_f(point[0], point[1])

# Different directions
angles = np.linspace(0, 2*np.pi, 8, endpoint=False)
directions = np.array([[np.cos(a), np.sin(a)] for a in angles])

# Compute directional derivatives
directional_derivs = [np.dot(gradient, d) for d in directions]

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Left: Directions on contour plot
contour = ax1.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
ax1.plot(point[0], point[1], 'ro', markersize=15, label='Point', zorder=5)

# Plot gradient
grad_norm = gradient / np.linalg.norm(gradient)
ax1.arrow(point[0], point[1], grad_norm[0]*0.5, grad_norm[1]*0.5,
         head_width=0.15, head_length=0.1, fc='red', ec='red', 
         linewidth=3, label='Gradient', zorder=4)

# Plot directions colored by derivative
colors = plt.cm.RdYlGn([(d - min(directional_derivs)) / 
                        (max(directional_derivs) - min(directional_derivs)) 
                        for d in directional_derivs])

for direction, deriv, color in zip(directions, directional_derivs, colors):
    ax1.arrow(point[0], point[1], direction[0]*0.4, direction[1]*0.4,
             head_width=0.1, head_length=0.08, 
             fc=color, ec='black', linewidth=1.5, alpha=0.8, zorder=3)

ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('Directional Derivatives in Different Directions', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.set_aspect('equal')

# Right: Bar plot of directional derivatives
bars = ax2.bar(range(len(directional_derivs)), directional_derivs, color=colors, edgecolor='black', linewidth=1.5)
ax2.axhline(y=0, color='k', linewidth=1)
ax2.set_xlabel('Direction', fontsize=12)
ax2.set_ylabel('Directional Derivative', fontsize=12)
ax2.set_title('Rate of Change in Each Direction', fontsize=13, fontweight='bold')
ax2.set_xticks(range(len(directional_derivs)))
ax2.set_xticklabels([f'{np.degrees(a):.0f}°' for a in angles], rotation=45)
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('🧭 Directional Derivatives', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"Gradient magnitude: {np.linalg.norm(gradient):.2f}")
print(f"Maximum directional derivative: {max(directional_derivs):.2f}")
print(f"Minimum directional derivative: {min(directional_derivs):.2f}")
print("\nThe gradient direction gives the maximum rate of increase!")

## 3. The Jacobian Matrix

When we have **multiple inputs and multiple outputs**, we use the Jacobian:

$$J = \begin{bmatrix}
\frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} \\
\frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2}
\end{bmatrix}$$

This appears in **every layer of a neural network**!

In [ ]:
# Vector-valued function: R^2 -> R^2
def vector_function(x, y):
    f1 = x**2 + y
    f2 = x + y**2
    return np.array([f1, f2])

# Jacobian matrix
def jacobian(x, y):
    return np.array([
        [2*x, 1],      # ∂f1/∂x, ∂f1/∂y
        [1, 2*y]       # ∂f2/∂x, ∂f2/∂y
    ])

# Visualize the transformation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Create grid of points
x_range = np.linspace(-2, 2, 15)
y_range = np.linspace(-2, 2, 15)
X_grid, Y_grid = np.meshgrid(x_range, y_range)

# Original space
ax1.scatter(X_grid, Y_grid, c='blue', alpha=0.5, s=30)
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('Input Space', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')
ax1.axhline(y=0, color='k', linewidth=0.5)
ax1.axvline(x=0, color='k', linewidth=0.5)

# Mark a special point
point = np.array([1.0, 0.5])
ax1.plot(point[0], point[1], 'r*', markersize=20, label=f'Point ({point[0]}, {point[1]})')
ax1.legend(fontsize=11)

# Transformed space
X_trans = np.zeros_like(X_grid)
Y_trans = np.zeros_like(Y_grid)

for i in range(X_grid.shape[0]):
    for j in range(X_grid.shape[1]):
        result = vector_function(X_grid[i,j], Y_grid[i,j])
        X_trans[i,j] = result[0]
        Y_trans[i,j] = result[1]

ax2.scatter(X_trans, Y_trans, c='red', alpha=0.5, s=30)
ax2.set_xlabel('f₁(x,y)', fontsize=12)
ax2.set_ylabel('f₂(x,y)', fontsize=12)
ax2.set_title('Output Space (After Transformation)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')
ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.axvline(x=0, color='k', linewidth=0.5)

# Mark transformed point
point_trans = vector_function(point[0], point[1])
ax2.plot(point_trans[0], point_trans[1], 'r*', markersize=20, 
         label=f'Transformed ({point_trans[0]:.1f}, {point_trans[1]:.1f})')
ax2.legend(fontsize=11)

plt.suptitle('Vector Function: Transforming Space', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Show Jacobian at the marked point
J = jacobian(point[0], point[1])
print(f"Jacobian matrix at point {point}:")
print(J)
print(f"\nDeterminant: {np.linalg.det(J):.2f} (measures area scaling)")

## 4. Multivariate Chain Rule

This is the **heart of backpropagation**!

If $z = f(y)$ and $y = g(x)$, then:

$$\frac{\partial z}{\partial x} = \frac{\partial z}{\partial y} \cdot \frac{\partial y}{\partial x}$$

In matrix form (for vectors):

$$\frac{\partial \mathbf{z}}{\partial \mathbf{x}} = \frac{\partial \mathbf{z}}{\partial \mathbf{y}} \cdot \frac{\partial \mathbf{y}}{\partial \mathbf{x}}$$

These are Jacobian matrices being multiplied!

In [ ]:
# Simple neural network example: x -> h -> y
# x (input) -> h = σ(W1*x + b1) -> y = W2*h + b2 (output)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Forward pass
def forward_pass(x, W1, b1, W2, b2):
    # First layer
    z1 = W1 @ x + b1
    h = sigmoid(z1)
    
    # Second layer
    z2 = W2 @ h + b2
    y = z2  # Linear output
    
    return y, h, z1

# Backpropagation: compute gradients
def backward_pass(x, y_true, y_pred, h, z1, W1, W2):
    # Loss: L = 0.5 * (y_pred - y_true)^2
    dL_dy = y_pred - y_true  # ∂L/∂y
    
    # Backprop through second layer
    dL_dW2 = dL_dy * h  # ∂L/∂W2 = ∂L/∂y * ∂y/∂W2
    dL_dh = W2 * dL_dy  # ∂L/∂h = ∂L/∂y * ∂y/∂h (chain rule!)
    
    # Backprop through sigmoid
    dL_dz1 = dL_dh * sigmoid_derivative(z1)  # ∂L/∂z1 = ∂L/∂h * ∂h/∂z1
    
    # Backprop through first layer
    dL_dW1 = dL_dz1 * x  # ∂L/∂W1 = ∂L/∂z1 * ∂z1/∂W1
    
    return dL_dW1, dL_dW2

# Example
np.random.seed(42)
x = np.array([1.0])
y_true = np.array([0.5])
W1 = np.array([[0.5]])
b1 = np.array([0.0])
W2 = np.array([[1.0]])
b2 = np.array([0.0])

# Forward
y_pred, h, z1 = forward_pass(x, W1, b1, W2, b2)
loss = 0.5 * (y_pred - y_true)**2

print("=" * 60)
print("FORWARD PASS")
print("=" * 60)
print(f"Input x: {x[0]:.3f}")
print(f"Hidden layer z1 = W1*x + b1: {z1[0]:.3f}")
print(f"Hidden activation h = σ(z1): {h[0]:.3f}")
print(f"Output y = W2*h + b2: {y_pred[0]:.3f}")
print(f"True value: {y_true[0]:.3f}")
print(f"Loss: {loss[0]:.6f}")

# Backward
dL_dW1, dL_dW2 = backward_pass(x, y_true, y_pred, h, z1, W1, W2)

print("\n" + "=" * 60)
print("BACKWARD PASS (Backpropagation via Chain Rule)")
print("=" * 60)
print(f"∂L/∂W2: {dL_dW2[0]:.6f}")
print(f"∂L/∂W1: {dL_dW1[0][0]:.6f}")
print("\nThese gradients tell us how to update the weights!")

### Visualizing Backpropagation

In [ ]:
# Create computational graph visualization
fig, ax = plt.subplots(figsize=(16, 10))
ax.axis('off')

# Node positions
positions = {
    'x': (1, 5),
    'z1': (3, 5),
    'h': (5, 5),
    'y': (7, 5),
    'L': (9, 5),
    'W1': (2, 7),
    'W2': (6, 7)
}

# Draw nodes
for name, (x_pos, y_pos) in positions.items():
    if name in ['W1', 'W2']:
        color = 'lightcoral'
        size = 1200
    elif name == 'L':
        color = 'gold'
        size = 1500
    else:
        color = 'lightblue'
        size = 1500
    
    ax.scatter(x_pos, y_pos, s=size, c=color, edgecolors='black', linewidths=2, zorder=3)
    ax.text(x_pos, y_pos, name, fontsize=16, fontweight='bold', 
            ha='center', va='center', zorder=4)

# Forward pass arrows (blue)
forward_arrows = [
    ('x', 'z1', 'W1·x'),
    ('z1', 'h', 'σ'),
    ('h', 'y', 'W2·h'),
    ('y', 'L', '(y-ŷ)²'),
    ('W1', 'z1', ''),
    ('W2', 'y', '')
]

for start, end, label in forward_arrows:
    x1, y1 = positions[start]
    x2, y2 = positions[end]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=3, color='blue', alpha=0.6))
    if label:
        mid_x, mid_y = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mid_x, mid_y + 0.3, label, fontsize=11, ha='center', 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Backward pass arrows (red)
backward_y = 3.5
backward_arrows = [
    ('L', 'y', '∂L/∂y'),
    ('y', 'h', '∂L/∂h'),
    ('h', 'z1', '∂L/∂z1'),
    ('z1', 'x', '∂L/∂x')
]

for start, end, label in backward_arrows:
    x1, y1 = positions[start]
    x2, y2 = positions[end]
    ax.annotate('', xy=(x2, backward_y), xytext=(x1, backward_y),
                arrowprops=dict(arrowstyle='<-', lw=3, color='red', alpha=0.7))
    mid_x = (x1 + x2) / 2
    ax.text(mid_x, backward_y - 0.4, label, fontsize=11, ha='center',
            color='red', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# Add title and legend
ax.text(5, 8.5, '🔄 Computational Graph: Forward & Backward Pass', 
        fontsize=18, fontweight='bold', ha='center')
ax.text(5, 1.5, 'Blue: Forward Pass (compute output) | Red: Backward Pass (compute gradients)', 
        fontsize=12, ha='center', style='italic')

# Add chain rule box
chain_rule_text = 'Chain Rule: ∂L/∂W1 = ∂L/∂z1 · ∂z1/∂W1'
ax.text(5, 0.5, chain_rule_text, fontsize=13, ha='center',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8, pad=0.8))

ax.set_xlim(0, 10)
ax.set_ylim(0, 9)
plt.tight_layout()
plt.show()

print("This is exactly what happens in EVERY neural network training step!")

## 5. The Hessian Matrix: Second Derivatives

The Hessian contains all second partial derivatives:

$$H = \begin{bmatrix}
\frac{\partial^2 f}{\partial x^2} & \frac{\partial^2 f}{\partial x \partial y} \\
\frac{\partial^2 f}{\partial y \partial x} & \frac{\partial^2 f}{\partial y^2}
\end{bmatrix}$$

**Uses:**
- Detecting saddle points vs. minima
- Second-order optimization (Newton's method)
- Understanding loss surface curvature

In [ ]:
# Function with interesting curvature
def f_hessian(x, y):
    return x**2 - y**2  # Saddle point at origin

def hessian_matrix(x, y):
    return np.array([
        [2, 0],   # ∂²f/∂x², ∂²f/∂x∂y
        [0, -2]   # ∂²f/∂y∂x, ∂²f/∂y²
    ])

# Visualize
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = f_hessian(X, Y)

fig = plt.figure(figsize=(18, 7))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='coolwarm', alpha=0.8, edgecolor='none')
ax1.scatter([0], [0], [0], color='red', s=200, marker='*', label='Saddle Point')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_zlabel('f(x,y)', fontsize=12)
ax1.set_title('f(x,y) = x² - y² (Saddle)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
fig.colorbar(surf, ax=ax1, shrink=0.5)

# Contour plot
ax2 = fig.add_subplot(122)
contour = ax2.contour(X, Y, Z, levels=20, cmap='coolwarm')
ax2.clabel(contour, inline=True, fontsize=10)
ax2.plot(0, 0, 'r*', markersize=20, label='Saddle Point (∇f = 0)')
ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('y', fontsize=12)
ax2.set_title('Contour Plot', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.suptitle('🎢 Saddle Point: Gradient is Zero but NOT a Minimum!', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Analyze with Hessian
H = hessian_matrix(0, 0)
eigenvalues = np.linalg.eigvals(H)

print("Hessian at (0, 0):")
print(H)
print(f"\nEigenvalues: {eigenvalues}")
print("\nInterpretation:")
print("• One positive eigenvalue → curves up in one direction")
print("• One negative eigenvalue → curves down in another direction")
print("• Mixed signs → SADDLE POINT (not a minimum!)")
print("\nThis is why gradient = 0 doesn't guarantee a minimum!")

## 6. Practical Example: Gradient Descent Step

In [ ]:
# Simple 2D optimization
def rosenbrock(x, y):
    """Rosenbrock function - classic optimization test"""
    return (1 - x)**2 + 100*(y - x**2)**2

def rosenbrock_grad(x, y):
    dx = -2*(1 - x) - 400*x*(y - x**2)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

# Gradient descent
def gradient_descent(start, learning_rate=0.001, iterations=1000):
    path = [start]
    point = start.copy()
    
    for i in range(iterations):
        grad = rosenbrock_grad(point[0], point[1])
        point = point - learning_rate * grad
        path.append(point.copy())
    
    return np.array(path)

# Run optimization
start_point = np.array([-1.0, 1.0])
path = gradient_descent(start_point, learning_rate=0.001, iterations=2000)

# Visualize
x = np.linspace(-2, 2, 200)
y = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(x, y)
Z = rosenbrock(X, Y)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Contour plot with path
contour = ax1.contourf(X, Y, np.log(Z + 1), levels=30, cmap='viridis', alpha=0.7)
contour_lines = ax1.contour(X, Y, np.log(Z + 1), levels=30, colors='black', 
                             alpha=0.2, linewidths=0.5)

# Plot gradient descent path
ax1.plot(path[:, 0], path[:, 1], 'r.-', linewidth=2, markersize=4, 
         alpha=0.7, label='Gradient Descent Path')
ax1.plot(path[0, 0], path[0, 1], 'go', markersize=15, label='Start', zorder=5)
ax1.plot(1, 1, 'r*', markersize=20, label='Global Minimum', zorder=5)
ax1.plot(path[-1, 0], path[-1, 1], 'bs', markersize=12, label='End', zorder=5)

ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('Gradient Descent on Rosenbrock Function', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
fig.colorbar(contour, ax=ax1)

# Loss over iterations
losses = [rosenbrock(p[0], p[1]) for p in path]
ax2.plot(losses, 'b-', linewidth=2)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Loss Decrease Over Time', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.suptitle('🎯 Gradient Descent in Action', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"Starting point: {start_point}")
print(f"Final point: {path[-1]}")
print(f"True minimum: [1, 1]")
print(f"\nStarting loss: {losses[0]:.2f}")
print(f"Final loss: {losses[-1]:.6f}")
print(f"Improvement: {(1 - losses[-1]/losses[0])*100:.2f}%")

## 🎯 Practice Exercises

In [ ]:
# Exercise 1: Compute the gradient of f(x,y) = x³ + xy + y²
def exercise_f(x, y):
    return x**3 + x*y + y**2

def exercise_grad(x, y):
    # TODO: Fill in the gradient!
    dx = 3*x**2 + y
    dy = x + 2*y
    return np.array([dx, dy])

# Visualize
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = exercise_f(X, Y)

fig, ax = plt.subplots(figsize=(12, 10))
contour = ax.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.7)

# Plot gradient field
step = 10
for i in range(0, len(x), step):
    for j in range(0, len(y), step):
        grad = exercise_grad(X[j, i], Y[j, i])
        grad_norm = np.linalg.norm(grad)
        if grad_norm > 0:
            grad_normalized = grad / grad_norm * 0.3
            ax.arrow(X[j, i], Y[j, i], grad_normalized[0], grad_normalized[1],
                    head_width=0.15, head_length=0.1, fc='red', ec='red', alpha=0.6)

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Your Gradient Field', fontsize=14, fontweight='bold')
fig.colorbar(contour)
plt.show()

test_point = (1, 1)
print(f"Gradient at {test_point}: {exercise_grad(*test_point)}")

## 🚀 Key Takeaways

1. **Gradient** = vector of partial derivatives, points toward steepest ascent
2. **Jacobian** = matrix of derivatives for vector-valued functions
3. **Chain rule** enables backpropagation through neural networks
4. **Hessian** captures curvature and helps identify saddle points
5. **Gradient descent** follows negative gradient to minimize loss
6. In neural networks:
   - Forward pass = compute outputs
   - Backward pass = compute gradients via chain rule
   - Update = move in negative gradient direction

## Next Steps

- Study matrix calculus notation
- Learn about automatic differentiation
- Implement backpropagation from scratch
- Explore advanced optimization algorithms (Adam, RMSprop)

### Resources
- "The Matrix Calculus You Need for Deep Learning" (Parr & Howard)
- 3Blue1Brown's neural networks series
- Stanford CS231n: Backpropagation notes